In [ ]:
import sys
import ast
import numpy as np
import pickle
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import pandas as pd
import os
import fnmatch

# 1989 year 
#year = 1989
#input_directories = [
# "/rdata/ian/pico/paperRuns/finalRuns/nsga2_None_1989_2024-10-15_14-58-22_p120",
# "/rdata/ian/pico/paperRuns/finalRuns/pinsga2_20to30_1989_2024-10-14_08-21-16_p60"
#]

#year = 1990
#input_directories = [
#    "/rdata/ian/pico/paperRuns/finalRuns/nsga2_None_1990_2024-10-16_15-28-33_p120",
#    "/rdata/ian/pico/paperRuns/finalRuns/pinsga2_20to30_1990_2024-10-16_07-41-32_p60"
#]


# 2000 year 
year = 2000
input_directories = ["/rdata/ian/pico/paperRuns/finalRuns/pinsga2_20to30_2000_2024-10-03_11-58-12_p60",
                     "/rdata/ian/pico/paperRuns/finalRuns/nsga2_None_2000_2024-08-29_16-04-56_p120"]

# 2001
#year = 2001
#input_directories = ["/rdata/ian/pico/paperRuns/finalRuns/pinsga2_20to30_2001_2024-09-26_15-35-40_p60",
#                     "/rdata/ian/pico/paperRuns/finalRuns/nsga2_None_2001_2024-09-05_09-37-42_p120"]

# 2002
#year = 2002
#input_directories = ["/rdata/ian/pico/paperRuns/finalRuns/nsga2_None_2002_2024-10-09_10-23-25_p120",
#                     "/rdata/ian/pico/paperRuns/finalRuns/pinsga2_20to30_2002_2024-10-08_15-22-06_p60"]

# 2003
#year = 2003
#input_directories = [
#    "/rdata/ian/pico/paperRuns/finalRuns/nsga2_None_2003_2024-10-10_18-51-49_p120",
#    "/rdata/ian/pico/paperRuns/finalRuns/pinsga2_20to30_2003_2024-10-10_10-03-58_p60"
#]



In [ ]:

input_directories = [directory.rstrip('/') for directory in input_directories]
#global_pf_file_name = r"/rdata/ian/pico/paperRuns/badDSSATruns/global_pf.pkl"
#
#global_pf = pd.read_pickle(global_pf_file_name)

output_dir = "/rdata/ian/pico/paperRuns/finalRuns"
output_file = "%s/global_run_table_%d.pkl" % (output_dir, year)



raw_master_table = {'year':[], 'yield':[], 'irr_total':[], 'front':[], 'irrigation':[], 'run':[], 'gen':[], 'algorithm':[]}


In [ ]:
def parse_directory_name(full_path):

    file_name = full_path.split("/")[-1]
    
    fields = file_name.split("_")

    result = {}
    
    result["algorithm"] = fields[0]
    result["DM_range"] = fields[1]
    result["year"] = int(fields[2])
    result["run_date"] = fields[3]
    result["run_time"] = fields[4]
    result["pop_size"] = fields[5][1:]

    return result

    

In [ ]:
def parse_file_name(file_name): 

    results = {}

    file_chunks = file_name.split("_")
    results["run"] = int(file_chunks[0][3:9])
    results["gen"] = int(file_chunks[1][3:9])

    return results 
    

In [ ]:
def get_file_names(directory, pattern): 

    matching_files = []
    for filename in os.listdir(directory):
        if fnmatch.fnmatch(filename, pattern):
            matching_files.append(os.path.join(directory, filename))
    
    return matching_files

In [ ]:
def read_files(files):

    all_solutions = None
    
    for (i, file_path) in enumerate(files):

        # Gather info from the file name
        file_name = file_path.split("/")[-1]
        full_dir = "/".join(file_path.split("/")[:-1])

        run_meta = parse_directory_name(full_dir)

        run_meta.update(parse_file_name(file_name) )
        # Get objective data
        current_objs = pd.read_csv(file_path, delimiter=',', names=["yield", "irr_total"])

        # Pull in metadata on the run        
        current_objs["yield"]     = current_objs["yield"] * -1
        current_objs["irr_total"] = current_objs["irr_total"] 
        current_objs["run"]       = int(run_meta["run"])
        current_objs["gen"]       = int(run_meta["gen"])
        current_objs["algorithm"] = run_meta["algorithm"] 
        current_objs["DM_range"]  = run_meta["DM_range"]  
        current_objs["year"]      = int(run_meta["year"])
        current_objs["run_date"]  = run_meta["run_date"]  
        current_objs["run_time"]  = run_meta["run_time"]  
        current_objs["pop_size"]  = int(run_meta["pop_size"])
                                    
        

        # Get decision variable data 
        var_file_path = file_path[:-7] + "var.csv"
        current_vars = pd.read_csv(var_file_path, delimiter=',', header=None)

        headers = ["var%s" % header for header in range(current_vars.shape[1])]
        current_vars = current_vars.set_axis(headers, axis=1)
        
        current_objs = pd.concat([current_objs,current_vars], axis=1)
        
        if all_solutions is None:
            all_solutions = current_objs
        else:
            all_solutions = pd.concat([all_solutions, current_objs])

    return all_solutions


### Read and accumulate data
Read each file individually and then add metadata on the run 

In [ ]:
results = None

for directory in input_directories:


    dir_meta = parse_directory_name(directory)

    year = dir_meta["year"]
        
    print("Processing year %s for folder %s" % (year, directory))

    obj_files = get_file_names(directory, 'run*obj.csv')

    if results is None: 
        results = read_files(obj_files)
    else: 
        current_results = read_files(obj_files)
        results = pd.concat([results,current_results])

    print("Successfully processed %d total items" % results.shape[0])


### Performance metrics
Measuring the performance of each configuration against a baseline 

In [ ]:

print("Picklin' the datar")

results.to_pickle(output_file)

print("Done")